# Tool Calling Fundamentals
### Practice Notebook

## 1. What is a tool, mechanically?

A tool is: a **name**, a **description** (tells the model *when* to use it),
and an **input schema** (tells the model *what arguments* it needs to
provide). The model never runs the function itself -- it only ever outputs a
*request* to call it; your code executes the real function and reports the
result back.

Let's define a real Python function, then wrap it with a tool schema in the
same shape the Anthropic API expects.


In [1]:
def get_weather(city: str, unit: str = "celsius") -> dict:
    """The REAL function that will actually run. In a production project
    this might call a weather API; here it returns fake data so the
    notebook works offline.
    """
    fake_temperatures = {"Bengaluru": 24, "Mumbai": 31, "Delhi": 19}
    temp_c = fake_temperatures.get(city, 25)
    if unit == "fahrenheit":
        return {"city": city, "temperature": temp_c * 9 / 5 + 32, "unit": "fahrenheit"}
    return {"city": city, "temperature": temp_c, "unit": "celsius"}

# The TOOL SCHEMA describing get_weather to an LLM
get_weather_tool = {
    "name": "get_weather",
    "description": (
        "Get the current weather for a specific city. Use this whenever the "
        "user asks about current temperature or weather conditions in a "
        "named city. Do not use this for weather forecasts (future dates)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city name, e.g. 'Bengaluru' or 'Mumbai'."
            },
            "unit": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit. Defaults to celsius if not specified."
            }
        },
        "required": ["city"]
    }
}

import json
print(json.dumps(get_weather_tool, indent=2))


{
  "name": "get_weather",
  "description": "Get the current weather for a specific city. Use this whenever the user asks about current temperature or weather conditions in a named city. Do not use this for weather forecasts (future dates).",
  "input_schema": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "The city name, e.g. 'Bengaluru' or 'Mumbai'."
      },
      "unit": {
        "type": "string",
        "enum": [
          "celsius",
          "fahrenheit"
        ],
        "description": "Temperature unit. Defaults to celsius if not specified."
      }
    },
    "required": [
      "city"
    ]
  }
}


**Exercise 2.1:** Identify the three things above that make this a *good*
tool definition: what specifically in the `description` tells the model when
*not* to use this tool? What would happen (in terms of model behavior) if
that sentence were deleted?


#### Exercise 2.1:

#### 1. Three things that make this a good tool definition:
* *Clear Description with Scope:* It explicitly explains what the tool does (gets current weather for a city) and when to use it.
* *Explicit Boundary Constraints (When NOT to use)*: It sets clear guardrails telling the model when to skip the tool ("Do not use this for weather forecasts (future dates).").
* *Strict Parameter Rules & Schema:* It specifies data types, defaults, exact options using an enum (["celsius", "fahrenheit"]), and marks city as a required parameter


#### 2. What sentence tells the model when *not* to use it?
"Do not use this for weather forecasts (future dates)."


#### 3. What would happen if that sentence were deleted?
* *Tool Misuse:* The model would try to call "get_weather" even when users ask for future predictions (e.g., *"What will the weather be in Mumbai next week?"*).
* *Wrong Data or Errors:* Since the tool only returns current live/fake data, the AI would either give inaccurate information for future dates or cause an error when trying to parse dates it doesn't support.

## 2. Simulating the model's tool-selection decision

We don't have a live model call here, so let's build a tiny **stand-in
router** that mimics what an LLM does when deciding which tool (if any) to
call, based on keyword matching. This is a crude approximation -- a real
LLM uses the full semantic content of the tool description and the user's
message, not keyword rules -- but it makes the *decision step* itself
visible and inspectable.


In [ ]:
import os

available_tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a specific city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name."}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount from one currency to another.",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Amount to convert"},
                    "from_currency": {"type": "string", "description": "Source currency code like USD"},
                    "to_currency": {"type": "string", "description": "Target currency code like INR"}
                },
                "required": ["amount", "from_currency", "to_currency"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_flights",
            "description": "Search for flights between two cities.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string"},
                    "destination": {"type": "string"}
                },
                "required": ["origin", "destination"]
            }
        }
    }
]

def llm_tool_router(user_message: str) -> str | None:
    """Simulates LLM tool routing locally to bypass Hugging Face 402 payment limits."""
    msg_lower = user_message.lower()
    
    if any(k in msg_lower for k in ["weather", "temperature", "delhi", "mumbai"]):
        return "get_weather"
    elif any(k in msg_lower for k in ["dollar", "rupee", "convert", "currency", "usd", "inr"]):
        return "convert_currency"
    elif any(k in msg_lower for k in ["flight", "fly", "bengaluru", "chennai"]):
        return "search_flights"
    
    return None

test_messages = [
    "What's the weather like in Delhi right now?",
    "How much is 100 US dollars in rupees?",
    "Find me a flight from Bengaluru to Chennai.",
    "Tell me a joke.",
]

for msg in test_messages:
    chosen = llm_tool_router(msg)
    print(f"'{msg}'  ->  tool: {chosen}")

'What's the weather like in Delhi right now?'  ->  tool: get_weather
'How much is 100 US dollars in rupees?'  ->  tool: convert_currency
'Find me a flight from Bengaluru to Chennai.'  ->  tool: search_flights
'Tell me a joke.'  ->  tool: None


**Exercise 2.2:** The last message ("Tell me a joke.") correctly returns
`None` -- no tool fits. Add a new message that's *ambiguous* between two
tools (e.g., something that mentions both weather and flights) and see which
one `stub_tool_router` picks. Is that a good decision? What does this reveal
about a real limitation of keyword-based routing that a real LLM (reasoning
over full context, not just keyword presence) would generally handle
better?


### Exercise 2.2 Answer

#### 1. Example Code
```python
ambiguous_msg = "Can you check the weather in Mumbai before I book a flight there?"
chosen = stub_tool_router(ambiguous_msg)
print(f"'{ambiguous_msg}'  ->  tool: {chosen}")

## 3. The full round-trip: call -> execute -> report result -> final answer

In [ ]:
import os
import re
import json
from huggingface_hub import InferenceClient

# 1. Real function
def get_weather(city: str, unit: str = "celsius") -> dict:
    fake_temperatures = {"Bengaluru": 24, "Mumbai": 31, "Delhi": 19}
    temp_c = fake_temperatures.get(city, 25)
    if unit == "fahrenheit":
        return {"city": city, "temperature": temp_c * 9 / 5 + 32, "unit": "fahrenheit"}
    return {"city": city, "temperature": temp_c, "unit": "celsius"}

# 2. Stub router
available_tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "keywords": ["weather", "temperature", "hot", "cold", "rain"],
    }
]

def stub_tool_router(user_message: str) -> str | None:
    msg_lower = user_message.lower()
    for tool in available_tools:
        if any(kw in msg_lower for kw in tool["keywords"]):
            return tool["name"]
    return None

# 3. Initialize Hugging Face Client
HF_TOKEN = os.environ.get("HF_TOKEN", "")
client = InferenceClient(api_key=HF_TOKEN)

# 4. Full tool use round trip function
def full_tool_use_round_trip(user_message: str):
    print(f"User: {user_message}")

    # Step 1: Model decides a tool call is needed
    tool_name = stub_tool_router(user_message)
    if tool_name is None:
        print("Assistant: (no tool needed) responding directly.")
        return

    # Step 2: Extract arguments
    city_match = re.search(r"in (\w+)", user_message)
    city = city_match.group(1) if city_match else "Bengaluru"

    print(f"Assistant requests tool call: {tool_name}(city='{city}')")

    # Step 3: YOUR CODE actually executes the function
    if tool_name == "get_weather":
        result = get_weather(city)
    else:
        result = {"error": f"no implementation wired up for {tool_name}"}

    print(f"Tool result returned to model: {result}")

    # Step 4: Real LLM call with error fallback
    messages = [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": f"Called tool {tool_name} with city='{city}'"},
        {"role": "user", "content": f"Tool result: {json.dumps(result)}. Provide a brief final answer."}
    ]

    try:
        response = client.chat.completions.create(
            model="mistralai/Mistral-7B-Instruct-v0.2",
            messages=messages,
            max_tokens=150
        )
        final_answer = response.choices[0].message.content
    except Exception as e:
        print(f"\n[Notice: API call skipped due to HF API status/token: {e}]")
        final_answer = f"The weather in {result.get('city', city)} is {result.get('temperature')}°C."

    print(f"Assistant (final): {final_answer}")

# Test the function
full_tool_use_round_trip("What's the weather like in Mumbai right now?")

User: What's the weather like in Mumbai right now?
Assistant requests tool call: get_weather(city='Mumbai')
Tool result returned to model: {'city': 'Mumbai', 'temperature': 31, 'unit': 'celsius'}

[Notice: API call skipped due to HF API status/token: (Request ID: Root=1-6acb9411-6a41c9b96c83e33d21f7754e;f6870e3b-e119-4e75-8df8-63a8e7791209)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.2' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}]
Assistant (final): The weather in Mumbai is 31°C.


**Exercise 2.3 (mini deliverable):** Implement a second real tool function
`convert_currency(amount, from_currency, to_currency)` (use a small hard-coded
exchange-rate dictionary, no real API needed) plus its tool schema (name,
description, input_schema) in the same style as `get_weather_tool`. Then
extend `full_tool_use_round_trip` so it actually calls your new tool for
currency-related messages, not just weather ones.


In [10]:
import json
import re

def get_weather(city: str, unit: str = "celsius") -> dict:
    fake_temperatures = {"Bengaluru": 24, "Mumbai": 31, "Delhi": 19}
    temp_c = fake_temperatures.get(city, 25)
    if unit == "fahrenheit":
        return {"city": city, "temperature": temp_c * 9 / 5 + 32, "unit": "fahrenheit"}
    return {"city": city, "temperature": temp_c, "unit": "celsius"}

def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    rates_to_usd = {
        "USD": 1.0,
        "INR": 0.012,
        "EUR": 1.08,
        "GBP": 1.27
    }
    
    from_curr = from_currency.upper()
    to_curr = to_currency.upper()
    
    if from_curr not in rates_to_usd or to_curr not in rates_to_usd:
        return {"error": f"Unsupported currency conversion between {from_curr} and {to_curr}"}
    
    
    amount_in_usd = amount * rates_to_usd[from_curr]
    converted_amount = round(amount_in_usd / rates_to_usd[to_curr], 2)
    
    return {
        "amount": amount,
        "from_currency": from_curr,
        "to_currency": to_curr,
        "converted_amount": converted_amount
    }

convert_currency_tool = {
    "name": "convert_currency",
    "description": (
        "Convert an amount from one fiat currency to another using current exchange rates. "
        "Use this whenever the user asks to calculate or convert money between currencies. "
        "Do not use this for cryptocurrency rates."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "amount": {
                "type": "number",
                "description": "The monetary amount to convert."
            },
            "from_currency": {
                "type": "string",
                "description": "3-letter ISO code for source currency, e.g. 'USD' or 'INR'."
            },
            "to_currency": {
                "type": "string",
                "description": "3-letter ISO code for target currency, e.g. 'INR' or 'EUR'."
            }
        },
        "required": ["amount", "from_currency", "to_currency"]
    }
}

available_tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "keywords": ["weather", "temperature", "hot", "cold", "rain"],
    },
    {
        "name": "convert_currency",
        "description": "Convert currency amounts.",
        "keywords": ["convert", "currency", "usd", "inr", "rupees", "dollars", "exchange"],
    }
]

def stub_tool_router(user_message: str) -> str | None:
    msg_lower = user_message.lower()
    for tool in available_tools:
        if any(kw in msg_lower for kw in tool["keywords"]):
            return tool["name"]
    return None

def full_tool_use_round_trip(user_message: str):
    print(f"\nUser: {user_message}")

    # Step 1: Model decides which tool call is needed
    tool_name = stub_tool_router(user_message)
    if tool_name is None:
        print("Assistant: (no tool needed) responding directly.")
        return

    # Step 2: Extract arguments based on selected tool
    if tool_name == "get_weather":
        city_match = re.search(r"in (\w+)", user_message)
        city = city_match.group(1) if city_match else "Bengaluru"
        print(f"Assistant requests tool call: {tool_name}(city='{city}')")
        
        # Step 3: Execute tool
        result = get_weather(city)
        print(f"Tool result returned to model: {result}")
        
        # Step 4: Final response
        print(f"Assistant (final): The weather in {result['city']} is {result['temperature']}°C.")

    elif tool_name == "convert_currency":
        # Simple extraction for demo: parse numbers and common currency keywords
        amount_match = re.search(r"\d+", user_message)
        amount = float(amount_match.group(0)) if amount_match else 100.0
        
        # Basic parsing check for USD/INR
        from_curr = "USD" if "us dollars" in user_message.lower() or "usd" in user_message.lower() else "INR"
        to_curr = "INR" if "rupees" in user_message.lower() or "inr" in user_message.lower() else "USD"
        
        print(f"Assistant requests tool call: {tool_name}(amount={amount}, from_currency='{from_curr}', to_currency='{to_curr}')")
        
        # Step 3: Execute tool
        result = convert_currency(amount, from_curr, to_curr)
        print(f"Tool result returned to model: {result}")
        
        # Step 4: Final response
        print(f"Assistant (final): {result['amount']} {result['from_currency']} is approximately {result['converted_amount']} {result['to_currency']}.")



full_tool_use_round_trip("What's the weather like in Mumbai right now?")
full_tool_use_round_trip("How much is 100 US dollars in rupees?")


User: What's the weather like in Mumbai right now?
Assistant requests tool call: get_weather(city='Mumbai')
Tool result returned to model: {'city': 'Mumbai', 'temperature': 31, 'unit': 'celsius'}
Assistant (final): The weather in Mumbai is 31°C.

User: How much is 100 US dollars in rupees?
Assistant requests tool call: convert_currency(amount=100.0, from_currency='USD', to_currency='INR')
Tool result returned to model: {'amount': 100.0, 'from_currency': 'USD', 'to_currency': 'INR', 'converted_amount': 8333.33}
Assistant (final): 100.0 USD is approximately 8333.33 INR.


## 4. (Optional) Wiring in a real LLM call

If you have an `ANTHROPIC_API_KEY` available, here's how the stub functions
above map onto the real Anthropic tool-use API. This cell is optional and
will not run without a valid key -- read through it even if you can't
execute it, since it shows exactly what the stubs above were standing in
for.


In [11]:
# client = InferenceClient(api_key=HF_TOKEN)
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     max_tokens=500,
#     tools=[get_weather_tool],
#     messages=[{"role": "user", "content": "What's the weather like in Mumbai right now?"}]
# )
#
# # The model's response will contain a tool_use block if it decided to call the tool:
# for block in response.choices[0].message.content:
#     if block.type == "tool_use":
#         print("Model wants to call:", block.name, "with input:", block.input)
#         tool_result = get_weather(**block.input)   # YOUR code actually runs it
#         # ... then send tool_result back in a new message with role='user',
#         # content=[{"type": "tool_result", "tool_use_id": block.id, "content": str(tool_result)}]

print("This cell is illustrative -- uncomment and add your API key to run it for real.")


This cell is illustrative -- uncomment and add your API key to run it for real.
